# 01 — Exploratory Data Analysis

**Course:** CSE2032 | Amazon ML Challenge | **50,000 curated benchmark pairs**

This notebook explores the benchmark dataset to understand class balance, name-length distributions, country coverage, and representative record pairs. Understanding the data before modelling is essential — it surfaces biases, informs feature design, and helps interpret model failures.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline
print("Setup complete.")

### 1. Load Benchmark Dataset

We load `data/processed/benchmark_pairs.csv`. Each row is a **pair** of business entities with columns `s1_*` (source record) and `probe_*` (candidate), plus `pair_type` and `label`.

In [ ]:
pairs = pd.read_csv('data/processed/benchmark_pairs.csv')
print(f'Shape: {pairs.shape}')
print(f'Columns: {list(pairs.columns)}')
pairs.head(5)

### 2. Class Balance & Pair Difficulty

The dataset contains three pair types:
- **positive** — genuine duplicates (label=1)
- **hard_negative** — similar-looking but different entities (label=0)
- **easy_negative** — clearly distinct entities (label=0)

Hard negatives are what distinguish a robust model from a naive one.

In [ ]:
label_counts = pairs['label'].value_counts().sort_index()
ptype_counts  = pairs['pair_type'].value_counts()

print('Label distribution:')
print(label_counts.to_string())
print('\nPair type distribution:')
print(ptype_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — label
bar_colors_lbl = ['#e74c3c', '#2ecc71']
axes[0].bar(label_counts.index.astype(str), label_counts.values,
            color=bar_colors_lbl, edgecolor='black', linewidth=0.7)
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label (0=Non-Match, 1=Match)')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Right — pair type
palette = {'positive': '#2ecc71', 'hard_negative': '#e67e22', 'easy_negative': '#3498db'}
bar_colors_pt = [palette.get(k, '#95a5a6') for k in ptype_counts.index]
axes[1].bar(ptype_counts.index, ptype_counts.values,
            color=bar_colors_pt, edgecolor='black', linewidth=0.7)
axes[1].set_title('Pair Type Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Pair Type')
axes[1].set_ylabel('Count')
for i, v in enumerate(ptype_counts.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.suptitle('Dataset Composition', y=1.03, fontsize=16, fontweight='bold')
import os; os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Name Length Analysis

Token count of business names reveals naming conventions. Very short names (e.g., "ABC") are ambiguous and harder to match; long descriptive names are more discriminative. We separate match vs. non-match distributions to see if length is a signal.

In [ ]:
pairs['s1_name_len']    = pairs['s1_name'].fillna('').str.split().str.len()
pairs['probe_name_len'] = pairs['probe_name'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_map = {1: ('Match', '#2ecc71'), 0: ('Non-Match', '#e74c3c')}
for lbl, (lname, color) in label_map.items():
    subset = pairs[pairs['label'] == lbl]
    sns.histplot(subset['s1_name_len'], ax=axes[0], label=lname,
                 color=color, alpha=0.55, bins=30, kde=True)
    sns.histplot(subset['probe_name_len'], ax=axes[1], label=lname,
                 color=color, alpha=0.55, bins=30, kde=True)

axes[0].set_title('s1_name Token Length', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Word Count'); axes[0].legend()
axes[1].set_title('probe_name Token Length', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count'); axes[1].legend()
plt.suptitle('Name Length Distribution by Match Label', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/eda_name_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. Country Distribution

The top countries help us understand geographic bias. Cross-country pairs also introduce script diversity (Latin, Devanagari, Arabic, CJK), which affects string similarity metrics.

In [ ]:
country_counts = pairs['s1_country'].value_counts().head(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(country_counts.index[::-1], country_counts.values[::-1],
                color=sns.color_palette('Blues_r', 10), edgecolor='black', linewidth=0.5)
plt.xlabel('Number of Records', fontsize=12)
plt.title('Top 10 Countries (s1_country)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, country_counts.values[::-1]):
    plt.text(val + 20, bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('results/plots/eda_country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5. Sample Record Pairs

Inspecting real examples builds intuition. Green rows are **positive** (true matches) and orange rows are **hard negatives** (similar names, different businesses — the model's hardest challenge).

In [ ]:
positives      = pairs[pairs['pair_type'] == 'positive'].head(3)
hard_negatives = pairs[pairs['pair_type'] == 'hard_negative'].head(3)

sample = pd.concat([positives, hard_negatives], ignore_index=True)
display_cols = ['s1_name', 'probe_name', 's1_country', 'probe_country', 'pair_type', 'label']
display_cols = [c for c in display_cols if c in sample.columns]
sample_display = sample[display_cols].copy()

def highlight_pair_type(row):
    if row['pair_type'] == 'positive':
        return ['background-color: #d5f5e3'] * len(row)
    elif row['pair_type'] == 'hard_negative':
        return ['background-color: #fde8d8'] * len(row)
    return [''] * len(row)

sample_display.style.apply(highlight_pair_type, axis=1)